# Week 12: Dynamic Programming (Memoization, 0/1 Knapsack, LCS & Edit Distance)

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

Top-down memoization, bottom-up tabulation, item backtracking, and string alignment.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-12/labs/W12_Dynamic_Programming.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Memoization vs. Tabulation & 0/1 Knapsack Backtracking

Unlike Fractional Knapsack, 0/1 Knapsack cannot be solved greedily by value-per-weight ratio: dynamic programming evaluates optimal subproblems across items and capacities in $O(nW)$.

In [1]:
def knapsack_01(capacity, items):
    n = len(items)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        val, wt = items[i - 1]
        for w in range(capacity + 1):
            dp[i][w] = dp[i - 1][w]
            if wt <= w and dp[i - 1][w - wt] + val > dp[i][w]:
                dp[i][w] = dp[i - 1][w - wt] + val
    chosen, w = [], capacity
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i - 1][w]:
            chosen.append(i - 1)
            w -= items[i - 1][1]
    return dp[n][capacity], chosen[::-1]


items = [(60, 10), (100, 20), (120, 30)]
best, picked = knapsack_01(50, items)
assert best == 220 and picked == [1, 2]
print(f"0/1 Knapsack best value = {best}, chosen indices = {picked} (greedy ratio pick would score only 160)")


0/1 Knapsack best value = 220, chosen indices = [1, 2] (greedy ratio pick would score only 160)


## 2. Longest Common Subsequence (LCS) & Levenshtein Edit Distance

2D dynamic programming tables compare prefixes $A[:i]$ and $B[:j]$ in $O(|A||B|)$ time for sequence alignment and spelling distance.

In [2]:
def lcs(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i - 1][j - 1] + 1 if a[i - 1] == b[j - 1] else max(dp[i - 1][j], dp[i][j - 1])
    out, i, j = [], m, n
    while i > 0 and j > 0:
        if a[i - 1] == b[j - 1]:
            out.append(a[i - 1]); i -= 1; j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return dp[m][n], "".join(reversed(out))


def edit_distance(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[m][n]


l_len, l_str = lcs("ZANZIBAR", "BANAIR")
ed = edit_distance("kitten", "sitting")
assert l_len == 4 and ed == 3
print(f"LCS('ZANZIBAR', 'BANAIR') = {l_str} (len {l_len}) | edit_distance('kitten', 'sitting') = {ed}")


LCS('ZANZIBAR', 'BANAIR') = ANIR (len 4) | edit_distance('kitten', 'sitting') = 3
